In [17]:
import sys
import os
import mysql.connector
import pandas as pd
import datetime
import random
import string
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

In [18]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: 3


In [19]:
cursor_old.execute("SHOW TABLES")
tables_data = cursor_old.fetchall()
target_tables = [list(t.values())[0] for t in tables_data]
print(f"\n--- Ditemukan {len(target_tables)} tabel di Database Lama ---")

df_old = {}
for table in target_tables:
    try:
        query = f"SELECT * FROM `{table}`"
        df_old[table] = pd.read_sql(query, db_old)
        print(f"Berhasil load tabel: {table} | Jumlah baris: {len(df_old[table])}")
    except Exception as e:
        print(f"Gagal load tabel {table}: {e}")

print("\n--- Proses load selesai. Semua data tersimpan di 'df_old' ---")


--- Ditemukan 108 tabel di Database Lama ---
Berhasil load tabel: absensi | Jumlah baris: 13444
Berhasil load tabel: absensi_note | Jumlah baris: 11
Berhasil load tabel: bidang | Jumlah baris: 4
Berhasil load tabel: bidangkategori | Jumlah baris: 12
Berhasil load tabel: bidanglink | Jumlah baris: 7
Berhasil load tabel: calon | Jumlah baris: 4
Berhasil load tabel: calon_detil | Jumlah baris: 61
Berhasil load tabel: calon_pertanyaan | Jumlah baris: 229
Berhasil load tabel: calon_pertanyaan_detil | Jumlah baris: 4305
Berhasil load tabel: catatan_kelas | Jumlah baris: 12797
Berhasil load tabel: catatan_kelas_tag | Jumlah baris: 999
Berhasil load tabel: catatan_mingguan | Jumlah baris: 0
Berhasil load tabel: catatan_siswa | Jumlah baris: 1502
Berhasil load tabel: catatan_siswa_follow_up | Jumlah baris: 22
Berhasil load tabel: catatanawal_admin | Jumlah baris: 64
Berhasil load tabel: catatanawal_datautama | Jumlah baris: 9
Berhasil load tabel: catatanawal_infolain | Jumlah baris: 64
Berhasi

In [20]:
cursor_new.execute("SHOW TABLES")
tables_data_new = cursor_new.fetchall()
target_tables_new = [list(t.values())[0] for t in tables_data_new]
df_new = {}

for table in target_tables_new:
    try:
        query = f"SELECT * FROM `{table}`"
        df_new[table] = pd.read_sql(query, db_new)
        print(f"Berhasil load tabel: {table} | Jumlah baris: {len(df_new[table])}")
    except:
        pass

Berhasil load tabel: absensi | Jumlah baris: 0
Berhasil load tabel: activity_log | Jumlah baris: 0
Berhasil load tabel: admin_sarpras | Jumlah baris: 1
Berhasil load tabel: bidang_kategori | Jumlah baris: 12
Berhasil load tabel: bidang_link | Jumlah baris: 7
Berhasil load tabel: busdev_bidang | Jumlah baris: 4
Berhasil load tabel: cache | Jumlah baris: 0
Berhasil load tabel: cache_locks | Jumlah baris: 0
Berhasil load tabel: calon_siswa | Jumlah baris: 160
Berhasil load tabel: calon_siswa_akademik | Jumlah baris: 158
Berhasil load tabel: calon_siswa_bayar | Jumlah baris: 166
Berhasil load tabel: calon_siswa_fo_detail | Jumlah baris: 0
Berhasil load tabel: calon_siswa_form_program_requirements | Jumlah baris: 0
Berhasil load tabel: calon_siswa_form_programs | Jumlah baris: 0
Berhasil load tabel: calon_siswa_jadwal | Jumlah baris: 166
Berhasil load tabel: calon_siswa_kursus | Jumlah baris: 166
Berhasil load tabel: calon_siswa_ortu | Jumlah baris: 166
Berhasil load tabel: calon_siswa_pros

In [21]:
# ==============================================================================
# CELL BARU UNTUK MERANGKUM HANYA TABEL TAMBAHAN (INDUK) YANG DIBUTUHKAN
# ==============================================================================

def rangkum_kebutuhan_tabel(cursor, nama_database, daftar_tabel):
    print("================================================================================")
    print(f" 📋 RANGKUMAN TABEL TAMBAHAN (INDUK) YANG HARUS DIMIGRASI 📋 ")
    print("================================================================================\n")
    
    if not daftar_tabel:
        print("Daftar tabel kosong.")
        return []

    # Menggunakan Set agar otomatis menghilangkan duplikat
    semua_tabel_dibutuhkan = set(daftar_tabel)
    tabel_antrean_cek = list(daftar_tabel)
    
    # Loop ini akan terus mencari "Induk dari Induk" sampai tidak ada lagi tabel baru
    while tabel_antrean_cek:
        format_tabel = ', '.join([f"'{t}'" for t in tabel_antrean_cek])
        
        query = f"""
        SELECT DISTINCT REFERENCED_TABLE_NAME
        FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
        WHERE TABLE_SCHEMA = '{nama_database}'
          AND TABLE_NAME IN ({format_tabel})
          AND REFERENCED_TABLE_NAME IS NOT NULL;
        """
        
        cursor.execute(query)
        hasil = cursor.fetchall()
        
        tabel_baru_ditemukan = []
        for row in hasil:
            # Handling jika cursor menggunakan mode dictionary atau tuple
            ref_table = row['REFERENCED_TABLE_NAME'] if isinstance(row, dict) else row[0]
            
            # Jika tabel referensi belum ada di penyimpanan utama, tambahkan!
            if ref_table not in semua_tabel_dibutuhkan:
                semua_tabel_dibutuhkan.add(ref_table)
                tabel_baru_ditemukan.append(ref_table)
        
        # Jadikan tabel yang baru ditemukan sebagai antrean cek selanjutnya
        tabel_antrean_cek = tabel_baru_ditemukan

    # === BAGIAN YANG DIUPDATE ===
    # Filter: Hanya ambil tabel yang TIDAK ADA di daftar_tabel awal
    tabel_hanya_tambahan = semua_tabel_dibutuhkan - set(daftar_tabel)
    
    # Ubah kembali Set ke format List dan urutkan sesuai abjad agar rapi
    daftar_tambahan_final = sorted(list(tabel_hanya_tambahan))
    
    print(f"Ditemukan {len(daftar_tambahan_final)} tabel tambahan (induk) yang belum ada di daftarmu:")
    
    if not daftar_tambahan_final:
        print("  ✅ Aman! Tidak ada tabel induk tambahan yang tercecer.")
    else:
        for i, tb in enumerate(daftar_tambahan_final, 1):
            print(f"  {i}. {tb}")
            
    print("\n💡 Variabel 'list_tabel_final' sekarang HANYA berisi tabel tambahan di atas.")
    return daftar_tambahan_final

# ==========================================
# EKSEKUSI RANGKUMAN
# ==========================================

# Ambil NAMA_DB_BARU dari konfigurasi (sesuaikan variabelmu)
# NAMA_DB_BARU = config["db_new"]["database"]

# Pastikan variabel daftar_yang_mau_dicek sudah dijalankan di cell atasnya
list_tabel_final = rangkum_kebutuhan_tabel(cursor_new, NAMA_DB_BARU, daftar_yang_mau_dicek)

# Melihat wujud akhir array-nya (Bisa di-copy paste)
print("\n[ WUJUD ARRAY MURNI TABEL TAMBAHAN ]:")
print(list_tabel_final)

 📋 RANGKUMAN TABEL TAMBAHAN (INDUK) YANG HARUS DIMIGRASI 📋 

Ditemukan 1 tabel tambahan (induk) yang belum ada di daftarmu:
  1. shift_kerja

💡 Variabel 'list_tabel_final' sekarang HANYA berisi tabel tambahan di atas.

[ WUJUD ARRAY MURNI TABEL TAMBAHAN ]:
['shift_kerja']


In [22]:
daftar_yang_mau_dicek = [
    'periode',
    'sesi',
    'libur',
    'kursus_libur',
    'kursus',
    'level',
    'kursus_level',
    'siswa',
    'kursus_siswa',
    'karyawan',
    'users',
    'jadwal',
    'jadwal_hari',
    'jadwal_detail',
    'jadwal_pengajar',
    'jadwal_siswa',
    'mitra',
    'calon_siswa',
    'calon_siswa_akademik',
    'calon_siswa_bayar',
    'calon_siswa_fo_detail',
    'calon_siswa_form_program_requirements',
    'calon_siswa_form_programs',
    'calon_siswa_jadwal',
    'calon_siswa_kursus',
    'calon_siswa_ortu',
    'calon_siswa_proses',
    'calon_siswa_proses_logs',
    'calon_siswa_status_logs',
    'catatan_kelas',
    'catatan_kelas_tag',
    'catatan_mingguan',
    'catatan_siswa',
    'followup_cs',
    'kabupaten',
    'kecamatan',
    'kelurahan',
    'kemitraan_verifikator',
    'kontak_prospek',
    'mitra_progres',
    'presensi_siswa',
    'provinsi',
    'siswa_keluar',
    'siswa_keluar_feedbacks',
    'siswa_mitra',
    'siswa_mitra_keluar',
    'tag_siswa_keluar',
    'topik_diskusi'
]

In [23]:
list_tabel_final = rangkum_kebutuhan_tabel(cursor_new, NAMA_DB_BARU, daftar_yang_mau_dicek)

print("\n[ WUJUD ARRAY MURNI ]:")
print(list_tabel_final)

 📋 RANGKUMAN TABEL TAMBAHAN (INDUK) YANG HARUS DIMIGRASI 📋 

Ditemukan 1 tabel tambahan (induk) yang belum ada di daftarmu:
  1. shift_kerja

💡 Variabel 'list_tabel_final' sekarang HANYA berisi tabel tambahan di atas.

[ WUJUD ARRAY MURNI ]:
['shift_kerja']
